In [1]:
from sklearnex import patch_sklearn
patch_sklearn()

import joblib
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss, precision_score, recall_score, f1_score
from tqdm import tqdm

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [2]:
print("Loading testing vectors...")
x_te = joblib.load('../vectors/x_testing_vector.pkl')
y_te = joblib.load('../vectors/y_testing_vector.pkl')
print("Loaded testing vectors")

Loading testing vectors...
Loaded testing vectors


In [3]:
print("Loading saved selector.")
sel = joblib.load('../models/sel.pkl')
print("Loaded saved selector")

Loading saved selector.
Loaded saved selector


In [4]:
print("Loading individual base models...")
model_names = [
    'Logistic Regression', 'SVM Poly', 'SVM RBF', 'SVM Linear', 'KNN', 'Random Forest', 'Ridge Classifier'
]

best_models = dict()

for name in model_names:
    print(f'Loading: {name}')
    best_models[name] = joblib.load(f'../models/model_{name}.pkl')
    print(f'Loaded: {name}')
    print()

print("Loaded individual base models.")

Loading individual base models...
Loading: Logistic Regression
Loaded: Logistic Regression

Loading: SVM Poly
Loaded: SVM Poly

Loading: SVM RBF
Loaded: SVM RBF

Loading: SVM Linear
Loaded: SVM Linear

Loading: KNN
Loaded: KNN

Loading: Random Forest
Loaded: Random Forest

Loading: Ridge Classifier
Loaded: Ridge Classifier

Loaded individual base models.


In [5]:
print("Applying feature selection to test data...")
x_te_s = sel.transform(x_te)

Applying feature selection to test data...


In [6]:
def evaluate_model(name, model, x_eval, y_eval):
    y_pred_local = model.predict(x_eval)
    y_prob_local = model.predict_proba(x_eval)

    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_eval, y_pred_local), 4),
        'Precision': round(precision_score(y_eval, y_pred_local), 4),
        'Recall': round(recall_score(y_eval, y_pred_local), 4),
        'F1': round(f1_score(y_eval, y_pred_local), 4),
        'Loss (Log Loss)': round(log_loss(y_eval, y_prob_local), 4),
    }

In [7]:
evaluation_rows = []
print('Testing models:', flush=True)
for model_name, model in tqdm(best_models.items(), total=len(best_models), desc='Testing models'):
    print(f'Testing model: {model_name}', flush=True)
    evaluation_rows.append(evaluate_model(model_name, model, x_te_s, y_te))
    print(f'Tested model: {model_name}')

evaluation_df = pd.DataFrame(evaluation_rows).sort_values(by='F1', ascending=False)


Testing models:


Testing models:   0%|                                                                            | 0/7 [00:00<?, ?it/s]

Testing model: Logistic Regression


Testing models:  14%|█████████▋                                                          | 1/7 [00:00<00:01,  5.35it/s]

Tested model: Logistic Regression
Testing model: SVM Poly


...

Tested model: KNN
Testing model: Random Forest


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 138 tasks      | elapsed:    1.3s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    1.9s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 138 tasks      | elapsed:    1.2s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    1.8s finished
Testing models:  86%|█████████████████████████████████████████████████████████▍         | 6/7 [31:46<07:27, 447.36s/it]

Tested model: Random Forest
Testing model: Ridge Classifier


Testing models: 100%|███████████████████████████████████████████████████████████████████| 7/7 [31:46<00:00, 272.35s/it]

Tested model: Ridge Classifier


In [8]:
print('Evaluation Matrix:')
print(evaluation_df.to_string(index=False))

Evaluation Matrix:
              Model  Accuracy  Precision  Recall     F1  Loss (Log Loss)
      Random Forest    0.9922     0.9889  0.9963 0.9926           0.0448
            SVM RBF    0.9848     0.9804  0.9908 0.9856           0.0488
                KNN    0.9835     0.9755  0.9936 0.9845           0.3066
           SVM Poly    0.9822     0.9736  0.9930 0.9832           0.0507
Logistic Regression    0.9693     0.9684  0.9734 0.9709           0.0805
         SVM Linear    0.9683     0.9633  0.9768 0.9700           0.0893
   Ridge Classifier    0.9642     0.9631  0.9690 0.9660           0.0963


In [9]:
evaluation_df.to_csv('../reports/model_report.csv', index=False)
print("Saved leaderboard to 'model_report.csv'")

Saved leaderboard to 'model_report.csv'


In [10]:
!jupyter nbconvert --to script model_testing.ipynb

[NbConvertApp] Converting notebook model_testing.ipynb to script
[NbConvertApp] Writing 2386 bytes to model_testing.py
